# SEA LEVEL PRESSURE $Climatology$ & $Anomaly$

In [1]:
import os
import numpy as np
import scipy as sp
import pandas as pd
from datetime import date
import marineHeatWaves as mhw
import netCDF4 as nc
import datetime
import matplotlib.pyplot as plt
from tqdm import notebook
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.backends.backend_agg import FigureCanvasAgg
import PIL.Image as Image
from scipy.stats import pearsonr
import matplotlib

In [2]:
slps=np.load('/lustre/home/yuhanxue/data/ERA/0.25area/re/new/msls.npy')
times=pd.to_datetime(np.load('/lustre/home/yuhanxue/data/ERA/0.25area/re/timess.npy',allow_pickle=True))

In [3]:
a=np.swapaxes(slps,0,1)
a=np.swapaxes(a,1,2)

In [4]:
times[-1]

Timestamp('2021-12-31 00:00:00')

In [5]:
t = np.arange(date(1993,1,1).toordinal(),date(2021,12,31).toordinal()+1)
def msls(dat):
    global t
    try:
        mhws, clim = mhw.detect(t,dat,climatologyPeriod=[1993,2021])
        return clim['seas']
    except:
        return np.array([np.nan]*10592)
def list_map(dat):
    pool = ProcessPoolExecutor(max_workers=9)
    ans=np.array(list(pool.map(msls,dat)))
    del pool
    return ans

In [6]:
pool = ProcessPoolExecutor(max_workers=6)
slp_c=np.array(list(pool.map(list_map,a)))
del pool


In [7]:
#np.savez('12_12_slp_clim.npz',slpclim=slp_c,time=times)

In [8]:
def getallname(s):
    sl=s.split()
    rs=''
    for i in sl:
        rs+=i
        rs+='_'
    return rs[:-1]

file=r"/lustre/home/yuhanxue/data/ERA/ERA5slp2021.nc"
daset=nc.Dataset(file)

names=[]
for i in daset.variables.keys():
    print(f'{i}:{getallname(daset.variables[i].long_name)} ({daset.variables[i].units})')
    names.append(getallname(daset.variables[i].long_name))
    #print(np.array(daset.variables[i]).shape)

longitude:longitude (degrees_east)
latitude:latitude (degrees_north)
time:time (hours since 1900-01-01 00:00:00.0)
t2m:2_metre_temperature (K)
msl:Mean_sea_level_pressure (Pa)


In [9]:
msl=np.array(daset['msl'])
msl=msl[:,::-1,:]
time=np.array(daset['time'])
lon=np.array(daset['longitude'])
lat=np.array(daset['latitude'])
lat=lat[::-1]
lat_ind=(lat>=35)&(lat<=50)
lon_ind=(lon>=-165)&(lon<=-130)
msl=msl[:,lat_ind,:][:,:,lon_ind]
lon=lon[lon_ind]
lat=lat[lat_ind]
Lon,Lat=np.meshgrid(lon,lat)
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))
time=pd.to_datetime(list(map(cftime2pdtime,nc.num2date(np.array(daset.variables['time']),daset.variables['time'].units))))

In [10]:
time2020ind=(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))

In [11]:
slp_c_2021=slp_c[:,:,time2020ind]
slp_2021=np.swapaxes(msl,0,1)
slp_2021=np.swapaxes(slp_2021,1,2)

In [12]:
slp_a=slp_2021-slp_c_2021

In [21]:
def fig2img(fig):
        """Convert a Matplotlib figure to a PIL Image and return it"""
        import io
        buf = io.BytesIO()
        fig.savefig(buf)
        buf.seek(0)
        img = Image.open(buf)
        return img
font_label = {'family': 'sans-serif',
                'size': 30}
font = {'family': 'sans-serif',
                'weight': 'normal'}
def draw(n):
    global slp_a,time,Lon,Lat
     
    matplotlib.use('Agg')
    plt.figure(figsize=[10,4],dpi=300)
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=0))
    plt.title(time[n].strftime('%y-%m-%d'))
    lon_formatter = LongitudeFormatter(zero_direction_label=False)
    lat_formatter = LatitudeFormatter()
    ax.xaxis.set_major_formatter(lon_formatter)
    ax.yaxis.set_major_formatter(lat_formatter)
    #scale = '50m'
    #land = cfeature.NaturalEarthFeature('physical', 'land', scale, edgecolor='face',facecolor=cfeature.COLORS['land'])
    #ax.add_feature(land, facecolor='0.75')
    #ax.coastlines()
    ax.set_xticks(range(-180, 131, 10))
    ax.set_yticks(range(35, 51, 5))
    c=plt.contourf(Lon,Lat,slp_a[:,:,n],np.arange(-3600,1601,100),cmap='RdBu_r')
    plt.colorbar(c,orientation='horizontal',label='Mean sea level pressure Anomaly (Pa)')
    image=fig2img(plt.gcf())
    return image

import warnings
warnings.filterwarnings("ignore")
images=list(map(draw,range(time.shape[0])))
im = images[0]
filename = 'test.gif'
im.save(fp=filename, format='gif', save_all=True, append_images=images[1:], duration=250,loop=0)
#draw(0)

In [ ]:
np.savez('slp_anomaly.npz',slp_anomaly=slp_a,lon=lon,lat=lat)